# 01 - Data Inspection

## Project
Spatial Patterns and Machine Learning Prediction of PM2.5 in London

## Research question
How do traffic exposure and green cover contribute to spatial variation in modelled PM2.5 concentrations across London, and how accurately can machine-learning models predict these concentrations?

This notebook inspects the source and prepared datasets used in the project.

In [1]:
from pathlib import Path
from zipfile import ZipFile
import sys

import geopandas as gpd
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CRS_EPSG, PROCESSED_DIR, RAW_DIR

print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DIR)
print("Processed data:", PROCESSED_DIR)
print("Working CRS: EPSG:", CRS_EPSG)

Project root: <project root>
Raw data: <project root>\data\raw
Processed data: <project root>\data\processed
Working CRS: EPSG: 27700


In [2]:
raw_files = sorted(RAW_DIR.iterdir())
print("Raw source files:\n")
for file in raw_files:
    if file.is_file():
        print(f"{file.name:65} {file.stat().st_size / 1024**2:,.2f} MB")

pm25_zip = RAW_DIR / "LAEI2022-Concentrations-Data-CSV.zip"
with ZipFile(pm25_zip, "r") as archive:
    csv_files = [name for name in archive.namelist() if name.lower().endswith(".csv")]
print("\nCSV files inside LAEI concentration archive:")
for name in csv_files:
    print("-", name)

Raw source files:

Green_Cover_2024.zip                                              288.89 MB
LAEI2022-Concentrations-Data-CSV.zip                              180.15 MB
LAEI2022-Concentrations-Data-ESRI.zip                             128.47 MB
LAEI2022-Traffic-Flows-and-Speeds-Major-and-Minor-Roads-GIS.zip   10.98 MB
London_Boroughs.gpkg                                              0.82 MB

CSV files inside LAEI concentration archive:
- CSV/LAEI2022_V1_NO2.csv
- CSV/LAEI2022_V1_NOx.csv
- CSV/LAEI2022_V1_PM10d.csv
- CSV/LAEI2022_V1_PM10m.csv
- CSV/LAEI2022_V1_PM25.csv


In [3]:
pm25_member = "CSV/LAEI2022_V1_PM25.csv"
with ZipFile(pm25_zip, "r") as archive:
    with archive.open(pm25_member) as file:
        pm25_sample = pd.read_csv(file, nrows=10, low_memory=False)

print("PM2.5 columns:")
print(pm25_sample.columns.tolist())
print("\nData types:")
print(pm25_sample.dtypes)
pm25_sample

PM2.5 columns:
['x', 'y', 'conc']

Data types:
x         int64
y         int64
conc    float64
dtype: object


,x,y,conc
0,519100,203440,7.892006
1,519120,203440,7.993147
2,519140,203440,8.219271
3,519160,203440,8.054134
4,519180,203440,7.904491
5,519200,203440,7.838168
6,519220,203440,7.800008
7,519240,203440,7.770976
8,519260,203440,7.747657
9,518940,203420,7.894738


In [4]:
traffic_dir = PROCESSED_DIR / "traffic"
major_roads = traffic_dir / "LAEI2022-major-roads-flows-and-speeds.gpkg"
minor_roads = traffic_dir / "LAEI2022-minor-roads-flows-and-speeds.gpkg"

print("Major-road layers:")
display(gpd.list_layers(major_roads))
print("Minor-road layers:")
display(gpd.list_layers(minor_roads))

major = gpd.read_file(major_roads, layer="major-roads-vehicle-flows-and-speeds")
minor = gpd.read_file(minor_roads, layer="minor-roads-vehicle-flows-and-speeds")
print("Major roads:", major.shape, "CRS:", major.crs)
print("Minor roads:", minor.shape, "CRS:", minor.crs)
print("\nMajor-road fields:", major.columns.tolist())
print("\nMinor-road fields:", minor.columns.tolist())

Major-road layers:


,name,geometry_type
0,major-roads-vehicle-flows-and-speeds,MultiLineString


Minor-road layers:


,name,geometry_type
0,minor-roads-vehicle-flows-and-speeds,MultiPolygon


Major roads: (79437, 19) CRS: EPSG:27700
Minor roads: (3460, 21) CRS: GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Longitude",EAST],AXIS["Latitude",NORTH]]

Major-road fields: ['year', 'TOID', 'laei.zone', 'borough', 'length.m', 'road.classification', 'speed.other.vehicles.kph', 'speed.buses.kph', 'AADF.motorcycles', 'AADF.taxis', 'AADF.cars', 'AADF.phvs', 'AADF.lgvs', 'AADF.hgvs.rigid', 'AADF.hgvs.articulated', 'AADF.tfl.buses', 'AADF.non.tfl.buses', 'AADF.coaches', 'geometry']

Minor-road fields: ['grid.id.unique', 'year', 'grid.id.1km2', 'easting', 'northing', 'area.km2', 'area.m2', 'laei.zone', 'borough', 'speed.kph', 'vkm.motorcycles', 'vkm.taxis', 'vkm.cars', 'vkm.phvs', 'vkm.lgvs', 'vkm.hgvs.rigid', 'vkm.hgvs.articulated', 'vkm.tfl.buses', 'vkm.non.tfl.buses', 'vkm.coaches', 'geometry']


In [5]:
grid_file = PROCESSED_DIR / "london_500m_grid.gpkg"
grid = gpd.read_file(grid_file, layer="london_500m_grid")
print("500 m grid shape:", grid.shape)
print("CRS:", grid.crs)
print("Bounds:", grid.total_bounds)

with ZipFile(RAW_DIR / "Green_Cover_2024.zip", "r") as archive:
    green_shapefiles = [name for name in archive.namelist() if name.lower().endswith(".shp")]
print("Green Cover shapefiles:", len(green_shapefiles))
print("First five tiles:")
for name in green_shapefiles[:5]:
    print("-", name)

500 m grid shape: (6672, 5)
CRS: EPSG:27700
Bounds: [503500. 155500. 562000. 201000.]
Green Cover shapefiles: 467
First five tiles:
- Green_Cover_2024/GreenCover24_100.shp
- Green_Cover_2024/GreenCover24_101.shp
- Green_Cover_2024/GreenCover24_102.shp
- Green_Cover_2024/GreenCover24_103.shp
- Green_Cover_2024/GreenCover24_104.shp
